# Coral Reef Health Detector - YOLOv5 (Client Version)

Run coral disease detection online using your trained YOLOv5 model. No heavy hardware required — Google provides the computing power.

### Step 1: Install Framework
Install the Ultralytics engine which supports both YOLOv5 and RT-DETR models.

In [ ]:
%pip install ultralytics opencv-python-headless
from ultralytics import YOLO
from IPython.display import display, Image
from google.colab import files
import os

### Step 2: Upload Your Test Image or Video
Run this cell and select the image (e.g. .jpg) or video (e.g. .mp4) you want to analyze.
The system assumes your `yolov5_best.pt` model is already placed in `/content/`.

In [ ]:
print("Please select your test image/video sequence now:")
uploaded = files.upload()
test_file = list(uploaded.keys())[0] if uploaded else None
if not test_file:
    print("ERROR: You forgot to upload a test image or video!")

### Step 3: Run Analysis
Loads your YOLOv5 model and scans the uploaded image or video. Annotated results will be displayed and downloaded automatically.

In [ ]:
import os
from ultralytics import YOLO
from IPython.display import display, Image
from google.colab import files

model_path = '/content/yolov5_best.pt'

if not os.path.exists(model_path):
    print(f"ERROR: I can't find your model at {model_path}! Please check the left sidebar.")
elif not test_file or not os.path.exists(test_file):
    print("ERROR: I can't find your test image or video! Please run Step 2 again.")
else:
    print(f"Analyzing '{test_file}' using '{model_path}'...\n")

    model = YOLO(model_path)
    results = model.predict(source=test_file, save=True, project="results", name="predict")

    print("\n" + "="*40)
    print("DETECTION SUMMARY (YOLOv5):")
    print("="*40)
    for box in results[0].boxes:
        class_id = int(box.cls[0])
        class_name = model.names[class_id]
        confidence = float(box.conf[0]) * 100
        if class_name == "Healthy Coral":
            print(f"[Healthy]     {class_name}: {confidence:.2f}%")
        elif "disease" in class_name.lower() or "pox" in class_name.lower():
            print(f"[Disease]     {class_name}: {confidence:.2f}%")
        elif "dead" in class_name.lower() or "bleached" in class_name.lower():
            print(f"[Warning]     {class_name}: {confidence:.2f}%")
        else:
            print(f"[Detected]    {class_name}: {confidence:.2f}%")
    print("="*40 + "\n")

    save_dir = results[0].save_dir
    final_result_path = os.path.join(save_dir, test_file)

    if os.path.exists(final_result_path):
        if final_result_path.lower().endswith(('.png', '.jpg', '.jpeg')):
            display(Image(filename=final_result_path))
        print("Prompting browser download of your annotated file...")
        files.download(final_result_path)
    else:
        print("Output file missing. Something went wrong with the save!")